# 🤖 02 - ACT Policy Training for Pick-Place

Train an **Action Chunking Transformer (ACT)** policy on MetaWorld `pick-place-v3` expert demonstrations.

**Prerequisites:** Run `01_pickplace_dataset.ipynb` first to generate the dataset.

**Key Hyperparameters:**
- `chunk_size=20` - Prevents mode collapse
- `batch_size=8` - GPU memory efficient
- `100k` training steps

In [ ]:
# ==========================================
# CELL 1: SYSTEM DEPENDENCIES
# ==========================================

!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
                         libosmesa6-dev software-properties-common patchelf

print("✅ System dependencies installed")

In [ ]:
# ==========================================
# CELL 2: ENVIRONMENT SETUP
# ==========================================

import os
import sys

os.environ['MUJOCO_GL'] = 'egl'
os.environ['LEROBOT_VIDEO_BACKEND'] = 'pyav'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Get Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print("✅ Secrets loaded")
except:
    print("⚠️ Set secrets manually if needed")

# Login to wandb
try:
    import wandb
    wandb.login(key=os.environ.get('WANDB_API_KEY', ''))
except:
    pass

print("✅ Environment configured")

In [ ]:
# ==========================================
# CELL 3: INSTALL PACKAGES
# ==========================================

!git clone https://github.com/huggingface/lerobot.git /kaggle/working/lerobot 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/lerobot
!pip install -e . -q
!pip install metaworld wandb imageio imageio-ffmpeg av -q

print("\n✅ All packages installed")

In [ ]:
# ==========================================
# CELL 4: CONFIGURATION
# ==========================================

import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/lerobot/src")

TASK_NAME = "pick-place-v3"
HF_USERNAME = "aryannzzz"  # <-- Change this!

# Dataset from 01_pickplace_dataset.ipynb
DATASET_REPO_ID = f"{HF_USERNAME}/metaworld-{TASK_NAME}-expert-v2"

# Training parameters
TRAINING_STEPS = 100000
BATCH_SIZE = 8
LEARNING_RATE = 0.0001
CHUNK_SIZE = 20  # IMPORTANT: Prevents mode collapse

# Output
OUTPUT_DIR = Path("/kaggle/working/outputs/pickplace")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POLICY_REPO_ID = f"{HF_USERNAME}/act-pick-place-v5"

print("📋 Training Configuration:")
print(f"   Task: {TASK_NAME}")
print(f"   Dataset: {DATASET_REPO_ID}")
print(f"   Steps: {TRAINING_STEPS:,}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Chunk size: {CHUNK_SIZE}")
print(f"   Output: {OUTPUT_DIR}")

---
## 🔍 DEBUG: Verify Dataset Before Training

In [ ]:
# ==========================================
# CELL 5: 🔍 DEBUG - Verify Dataset
# ==========================================

from huggingface_hub import list_repo_files

print(f"🔍 Verifying dataset: {DATASET_REPO_ID}\n")

try:
    files = list_repo_files(DATASET_REPO_ID, repo_type="dataset")
    
    # Check for required LeRobot metadata
    required = ['meta/info.json', 'meta/stats.json', 'meta/episodes.jsonl']
    missing = [f for f in required if f not in files]
    
    print("📁 Key files:")
    for f in required:
        status = "✅" if f in files else "❌"
        print(f"   {status} {f}")
    
    if missing:
        print(f"\n❌ Missing metadata files! Re-run 01_pickplace_dataset.ipynb")
    else:
        print(f"\n✅ Dataset verified! Ready to load.")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n⚠️ Make sure to run 01_pickplace_dataset.ipynb first!")

In [ ]:
# ==========================================
# CELL 6: LOAD DATASET
# ==========================================

from lerobot.datasets.lerobot_dataset import LeRobotDataset

print(f"📦 Loading dataset: {DATASET_REPO_ID}")

dataset = LeRobotDataset(
    repo_id=DATASET_REPO_ID,
    video_backend="pyav"
)

print(f"\n📊 Dataset loaded:")
print(f"   Episodes: {dataset.num_episodes}")
print(f"   Frames: {dataset.num_frames:,}")
print(f"   FPS: {dataset.fps}")
print(f"   Features: {list(dataset.features.keys())}")

In [ ]:
# ==========================================
# CELL 7: 🔍 DEBUG - Check Action Statistics
# ==========================================

import numpy as np

print("🔍 Checking action statistics...\n")

# Sample actions from dataset
actions = []
for i in range(min(500, dataset.num_frames)):
    sample = dataset[i]
    actions.append(sample['action'].numpy())

actions = np.array(actions)
print(f"📊 Action statistics (sampled {len(actions)} frames):")
print(f"   Shape: {actions.shape}")
print(f"   Mean: {np.mean(actions, axis=0)}")
print(f"   Std:  {np.std(actions, axis=0)}")
print(f"   Min:  {np.min(actions, axis=0)}")
print(f"   Max:  {np.max(actions, axis=0)}")

if np.all(np.std(actions, axis=0) > 0.05):
    print("\n✅ Actions have good variance!")
else:
    print("\n⚠️ Some action dimensions have low variance")

---
## 🚀 Training

In [ ]:
# ==========================================
# CELL 8: BUILD ACT POLICY CONFIG
# ==========================================

from lerobot.configs.policies import ACTConfig
from lerobot.models.act.configuration_act import ACTConfig as ACTModelConfig

print("🔧 Building ACT policy configuration...")

# Get observation/action shapes from dataset
sample = dataset[0]
state_dim = sample['observation.state'].shape[0]
action_dim = sample['action'].shape[0]

print(f"   State dim: {state_dim}")
print(f"   Action dim: {action_dim}")

# Create ACT config
policy_cfg = ACTConfig(
    input_shapes={
        "observation.images.image": [3, 480, 480],
        "observation.state": [state_dim],
    },
    output_shapes={
        "action": [action_dim],
    },
    input_normalization_modes={
        "observation.images.image": "mean_std",
        "observation.state": "mean_std",
    },
    output_normalization_modes={
        "action": "mean_std",
    },
    chunk_size=CHUNK_SIZE,
    n_action_steps=CHUNK_SIZE,
)

print(f"\n✅ ACT config created")
print(f"   Chunk size: {policy_cfg.chunk_size}")

In [ ]:
# ==========================================
# CELL 9: CREATE POLICY
# ==========================================

from lerobot.policies.act.modeling_act import ACTPolicy
import torch

print("🤖 Creating ACT policy...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

policy = ACTPolicy(
    config=policy_cfg,
    dataset_stats=dataset.meta.stats
)
policy.to(device)

# Count parameters
total_params = sum(p.numel() for p in policy.parameters())
trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)

print(f"\n✅ Policy created")
print(f"   Total params: {total_params:,}")
print(f"   Trainable: {trainable_params:,}")

In [ ]:
# ==========================================
# CELL 10: TRAINING LOOP
# ==========================================

from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import wandb

print(f"\n{'='*70}")
print(f"🚀 Starting Training")
print(f"{'='*70}")
print(f"   Steps: {TRAINING_STEPS:,}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"{'='*70}\n")

# Initialize wandb
try:
    wandb.init(
        project="metaworld-act",
        name=f"pick-place-v5-{CHUNK_SIZE}chunk",
        config={
            "task": TASK_NAME,
            "dataset": DATASET_REPO_ID,
            "batch_size": BATCH_SIZE,
            "chunk_size": CHUNK_SIZE,
            "learning_rate": LEARNING_RATE,
            "training_steps": TRAINING_STEPS,
        }
    )
    use_wandb = True
except:
    use_wandb = False
    print("⚠️ W&B not available, logging locally only")

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)

# Optimizer
optimizer = AdamW(policy.parameters(), lr=LEARNING_RATE)

# Training
policy.train()
step = 0
running_loss = 0.0
log_interval = 100
save_interval = 10000

pbar = tqdm(total=TRAINING_STEPS, desc="Training")

while step < TRAINING_STEPS:
    for batch in dataloader:
        if step >= TRAINING_STEPS:
            break
        
        # Move batch to device
        batch = {k: v.to(device) if hasattr(v, 'to') else v for k, v in batch.items()}
        
        # Forward pass
        loss_dict = policy.forward(batch)
        loss = loss_dict["loss"]
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        step += 1
        pbar.update(1)
        
        # Log
        if step % log_interval == 0:
            avg_loss = running_loss / log_interval
            pbar.set_postfix({"loss": f"{avg_loss:.4f}"})
            
            if use_wandb:
                wandb.log({"loss": avg_loss, "step": step})
            
            running_loss = 0.0
        
        # Save checkpoint
        if step % save_interval == 0:
            ckpt_path = OUTPUT_DIR / f"checkpoint_{step}.pt"
            torch.save({
                "step": step,
                "model_state_dict": policy.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            }, ckpt_path)
            print(f"\n💾 Saved checkpoint: {ckpt_path}")

pbar.close()

print(f"\n{'='*70}")
print(f"✅ Training Complete!")
print(f"{'='*70}")

In [ ]:
# ==========================================
# CELL 11: SAVE FINAL MODEL
# ==========================================

print("💾 Saving final model...")

# Save locally
final_path = OUTPUT_DIR / "policy_final.pt"
torch.save({
    "step": step,
    "model_state_dict": policy.state_dict(),
    "config": policy_cfg,
}, final_path)
print(f"   Saved to: {final_path}")

# Save for LeRobot loading
policy.save_pretrained(str(OUTPUT_DIR / "pretrained"))
print(f"   Saved pretrained to: {OUTPUT_DIR / 'pretrained'}")

print("\n✅ Model saved!")

In [ ]:
# ==========================================
# CELL 12: UPLOAD TO HUGGINGFACE
# ==========================================

print(f"📤 Uploading policy to HuggingFace: {POLICY_REPO_ID}")

try:
    policy.push_to_hub(
        repo_id=POLICY_REPO_ID,
        commit_message=f"ACT policy trained on {DATASET_REPO_ID}",
        tags=["metaworld", "pick-place", "act", "lerobot"],
    )
    print(f"\n✅ Uploaded to: https://huggingface.co/{POLICY_REPO_ID}")
except Exception as e:
    print(f"❌ Upload failed: {e}")
    print("\n⚠️ Model saved locally. You can upload manually.")

---
## 🎮 Evaluation

In [ ]:
# ==========================================
# CELL 13: EVALUATE POLICY
# ==========================================

from lerobot.envs.metaworld import MetaworldEnv
import numpy as np

print("🎮 Evaluating trained policy...\n")

# Create environment
env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=480,
    observation_height=480,
)

policy.eval()
policy.reset()

NUM_EVAL_EPISODES = 10
successes = 0
rewards_all = []

for ep in range(NUM_EVAL_EPISODES):
    obs, info = env.reset(seed=ep * 100)
    policy.reset()
    
    done = False
    ep_reward = 0
    ep_success = False
    
    for step in range(500):
        # Prepare observation for policy
        obs_dict = {
            "observation.images.image": torch.from_numpy(obs["pixels"]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0,
            "observation.state": torch.from_numpy(obs["agent_pos"]).unsqueeze(0).float().to(device),
        }
        
        with torch.no_grad():
            action = policy.select_action(obs_dict)
        
        action_np = action.squeeze().cpu().numpy()
        obs, reward, terminated, truncated, info = env.step(action_np)
        
        ep_reward += reward
        if info.get("success", False) or info.get("is_success", False):
            ep_success = True
        
        if terminated or truncated:
            break
    
    if ep_success:
        successes += 1
    rewards_all.append(ep_reward)
    
    print(f"   Episode {ep+1}: Reward={ep_reward:.1f}, Success={ep_success}")

env.close()

print(f"\n{'='*70}")
print(f"📊 Evaluation Results")
print(f"{'='*70}")
print(f"   Success rate: {100 * successes / NUM_EVAL_EPISODES:.1f}%")
print(f"   Avg reward: {np.mean(rewards_all):.1f}")
print(f"{'='*70}")

---
## ✅ Training Complete!

**Next Steps:**
1. Evaluate the policy on more episodes
2. Run `03_handlepull_dataset.ipynb` to create handle-pull dataset
3. Run `04_handlepull_train.ipynb` to train on handle-pull
4. Use `05_multitask_act.ipynb` for multi-task learning